In [5]:
import shutil
from pathlib import Path

def find_path(pattern):
    matches = sorted(Path("/kaggle/input").glob(pattern))
    if not matches:
        raise FileNotFoundError(f"Could not find pattern: {pattern}")
    return matches[0]

REPO_SRC = find_path("**/VIIS_patched_v13")
BASE_CKPT = find_path("**/datasets/rushikeshkulkarni401/required-weights/required_weights/viis_slim.ckpt")
V13_CKPT = find_path("**/datasets/rushikeshkulkarni401/required-weights/required_weights/v13/best.pt")
V13_2_CKPT = find_path("**/datasets/rushikeshkulkarni401/required-weights/required_weights/v13.2/best.pt")
MSRS_DIR = find_path("**/datasets/elixir4o4/msrs-main")
LLVIP_DIR = find_path("**/datasets/afradhossain/llvip-dataset")

REPO_WORK = Path("/kaggle/working/VIIS_patched_v13")
if REPO_WORK.exists():
    shutil.rmtree(REPO_WORK)
shutil.copytree(REPO_SRC, REPO_WORK)

print("REPO_WORK :", REPO_WORK)
print("BASE_CKPT :", BASE_CKPT)
print("V13_CKPT  :", V13_CKPT)
print("V13_2_CKPT:", V13_2_CKPT)
print("MSRS_DIR  :", MSRS_DIR)
print("LLVIP_DIR :", LLVIP_DIR)

REPO_WORK : /kaggle/working/VIIS_patched_v13
BASE_CKPT : /kaggle/input/datasets/rushikeshkulkarni401/required-weights/required_weights/viis_slim.ckpt
V13_CKPT  : /kaggle/input/datasets/rushikeshkulkarni401/required-weights/required_weights/v13/best.pt
V13_2_CKPT: /kaggle/input/datasets/rushikeshkulkarni401/required-weights/required_weights/v13.2/best.pt
MSRS_DIR  : /kaggle/input/datasets/elixir4o4/msrs-main
LLVIP_DIR : /kaggle/input/datasets/afradhossain/llvip-dataset


In [9]:
import os, random
from pathlib import Path

MSRS_TEST = next(Path("/kaggle/input").glob("**/datasets/elixir4o4/msrs-main/MSRS-main/test"))
SUBSET_ROOT = Path("/kaggle/working/msrs_eval_20/test")

rgb_name = "vi" if (MSRS_TEST / "vi").exists() else "rgb"
rgb_dir = MSRS_TEST / rgb_name
ir_dir = MSRS_TEST / "ir"

all_files = sorted([f.name for f in rgb_dir.iterdir() if f.is_file()])
day_files = [f for f in all_files if "D" in Path(f).stem]
night_files = [f for f in all_files if "N" in Path(f).stem]

# Exclude every 5th MSRS-test Day image (reserved for validation in v13.1)
heldout_day = [f for idx, f in enumerate(day_files) if idx % 5 != 0]

rng = random.Random(123)
selected_day = sorted(rng.sample(heldout_day, 20))
selected_night = sorted(rng.sample(night_files, 20))

for sub in [rgb_name, "ir"]:
    (SUBSET_ROOT / sub).mkdir(parents=True, exist_ok=True)

for fname in selected_day + selected_night:
    os.symlink(rgb_dir / fname, SUBSET_ROOT / rgb_name / fname)
    os.symlink(ir_dir / fname, SUBSET_ROOT / "ir" / fname)

print(f"Created {len(selected_day)} Day + {len(selected_night)} Night pairs in {SUBSET_ROOT.parent}")

FileExistsError: [Errno 17] File exists: '/kaggle/input/datasets/elixir4o4/msrs-main/MSRS-main/test/vi/00081D.png' -> '/kaggle/working/msrs_eval_20/test/vi/00081D.png'

In [11]:
%%bash
cd /kaggle/working/VIIS_patched_v13
cat trg13/paths.py
echo "----------------------------------------"
grep -n -C 5 "add_argument" infer_v13.py

"""Dataset / checkpoint discovery. Override anything with env vars MSRS, LLVIP, VIIS_CKPT, RUNS."""
import os

_SKIP = {"vi", "ir", "visible", "infrared", "annotations", "label", "labels", "__pycache__", ".git"}
DEFAULTS = dict(
    MSRS="/kaggle/input/datasets/elixir4o4/msrs-main/MSRS-main",
    LLVIP="/kaggle/input/datasets/afradhossain/llvip-dataset/LLVIP",
    RUNS="/kaggle/working/runs",
)
SLIM_DEFAULT = "/kaggle/working/viis_slim.ckpt" if os.path.isdir("/kaggle/working") else os.path.abspath("viis_slim.ckpt")


def _walk(base, max_depth=7):
    if not os.path.isdir(base):
        return
    base = os.path.abspath(base)
    for dp, dns, fns in os.walk(base):
        depth = dp[len(base):].count(os.sep)
        dns[:] = [d for d in sorted(dns) if d.lower() not in _SKIP] if depth < max_depth else []
        yield dp, dns, fns


def find_msrs(bases=("/kaggle/input", "/kaggle/working")):
    for b in bases:
        for dp, _, _ in _walk(b):
            if os.path.isdir(f"{dp}/train/vi

In [13]:
import os, time, subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/VIIS_patched_v13")

# 1. Resolve exact dataset & checkpoint paths
MSRS_ROOT = next(p.parent.parent for p in Path("/kaggle/input").glob("**/train/vi"))
LLVIP_ROOT = next(p.parent.parent for p in Path("/kaggle/input").glob("**/visible/train"))
BASE_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/viis_slim.ckpt"))
V13_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/v13/best.pt"))
V13_2_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/v13.2/best.pt"))

# 2. Verify paths via trg13/paths.py
base_env = os.environ.copy()
base_env["MSRS"] = str(MSRS_ROOT)
base_env["LLVIP"] = str(LLVIP_ROOT)
base_env["VIIS_CKPT"] = str(BASE_CKPT)
base_env["PYTHONUNBUFFERED"] = "1"

print("=== Path Verification ===")
subprocess.run(["python", "trg13/paths.py"], cwd=WORK_DIR, env=base_env, check=True)
print("=========================\n")

OUT_V13 = Path("/kaggle/working/eval_v13_full")
OUT_V13_2 = Path("/kaggle/working/eval_v13_2_full")
OUT_V13.mkdir(parents=True, exist_ok=True)
OUT_V13_2.mkdir(parents=True, exist_ok=True)

# 20 Day images x 3 severities (light/low/dark) + 20 Night images = 80 inferences per model
TOTAL_EXPECTED = 80

def launch(gpu_id, ckpt_path, label, out_dir, log_path):
    env = base_env.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [
        "python", "-u", "infer_v13.py",
        "--ckpts", str(ckpt_path),
        "--labels", label,
        "--out", str(out_dir),
        "--zeroshot", "0",
        "--sets", "msrs_syn,real",
        "--n_msrs", "20",
        "--n_msrs_night", "20",
        "--n_llvip", "0",
        "--steps", "25",
        "--long_side", "512",
        "--seed", "123",
    ]
    log_f = open(log_path, "w")
    proc = subprocess.Popen(cmd, cwd=WORK_DIR, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    return proc, log_f

def last_line(log_path):
    try:
        lines = Path(log_path).read_text(errors="ignore").splitlines()
        return lines[-1].strip()[-80:] if lines else "starting..."
    except Exception:
        return "starting..."

def count_pngs(out_dir):
    return len(list(out_dir.rglob("*.png")))

log1, log2 = "/kaggle/working/v13_full.log", "/kaggle/working/v13_2_full.log"
p0, f0 = launch(0, V13_CKPT, "v13_full", OUT_V13, log1)
p1, f1 = launch(1, V13_2_CKPT, "v13_2_full", OUT_V13_2, log2)

t0 = time.time()
print("Started parallel inference on GPU 0 (v13) and GPU 1 (v13.2)...\n")

while True:
    r0, r1 = p0.poll(), p1.poll()
    done0, done1 = count_pngs(OUT_V13), count_pngs(OUT_V13_2)
    pend0, pend1 = max(0, TOTAL_EXPECTED - done0), max(0, TOTAL_EXPECTED - done1)
    elapsed = int(time.time() - t0)
    mins, secs = divmod(elapsed, 60)

    s0 = f"DONE (code {r0})" if r0 is not None else f"RUNNING | PNGs: {done0}/{TOTAL_EXPECTED} (pending: {pend0})"
    s1 = f"DONE (code {r1})" if r1 is not None else f"RUNNING | PNGs: {done1}/{TOTAL_EXPECTED} (pending: {pend1})"

    print(f"[{mins:02d}m {secs:02d}s]")
    print(f"  GPU 0 [v13  ]: {s0} | last: {last_line(log1)}")
    print(f"  GPU 1 [v13.2]: {s1} | last: {last_line(log2)}")

    if r0 is not None and r1 is not None:
        break
    time.sleep(20)

f0.close()
f1.close()
print("\nBoth inference runs finished.")

=== Path Verification ===
MSRS   /kaggle/input/datasets/elixir4o4/msrs-main/MSRS-main  OK
LLVIP  /kaggle/input/datasets/afradhossain/llvip-dataset/LLVIP  OK
CKPT   /kaggle/input/datasets/rushikeshkulkarni401/required-weights/required_weights/viis_slim.ckpt  OK
RUNS   /kaggle/working/runs  (created on first run)

Started parallel inference on GPU 0 (v13) and GPU 1 (v13.2)...

 00s]
  GPU 0 [v13  ]: RUNNING | PNGs: 0/80 (pending: 80) | last: starting...
  GPU 1 [v13.2]: RUNNING | PNGs: 0/80 (pending: 80) | last: starting...
 20s]
  GPU 0 [v13  ]: RUNNING | PNGs: 220/80 (pending: 0) | last: [    13s] items: msrs_syn=60, real=20
  GPU 1 [v13.2]: RUNNING | PNGs: 220/80 (pending: 0) | last: [    12s] items: msrs_syn=60, real=20
 40s]
  GPU 0 [v13  ]: RUNNING | PNGs: 220/80 (pending: 0) | last: [    13s] items: msrs_syn=60, real=20
  GPU 1 [v13.2]: RUNNING | PNGs: 220/80 (pending: 0) | last: [    12s] items: msrs_syn=60, real=20
 00s]
  GPU 0 [v13  ]: RUNNING | PNGs: 220/80 (pending: 0) | las

In [14]:
import math
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

EVAL_V13 = Path("/kaggle/working/eval_v13_full")
EVAL_V13_2 = Path("/kaggle/working/eval_v13_2_full")
OUT_PDF = Path("/kaggle/working/v13_vs_v13_2_comparison.pdf")

ROWS_PER_PAGE = 4  # 4 scenes per landscape page keeps every image large and sharp


def build_section_pages(pdf, title, rows_data, col_headers):
    """Renders multi-page comparison grids at high DPI without blurring."""
    n_cols = len(col_headers)
    total_pages = math.ceil(len(rows_data) / ROWS_PER_PAGE)

    for page_idx in range(total_pages):
        batch = rows_data[page_idx * ROWS_PER_PAGE : (page_idx + 1) * ROWS_PER_PAGE]
        n_rows = len(batch)

        # Size figure proportionally to image aspect ratio (640x480 -> 4:3)
        fig_w = 4.2 * n_cols
        fig_h = 3.2 * n_rows + 0.9
        fig, axes = plt.subplots(
            n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False
        )
        fig.suptitle(
            f"{title}  (Page {page_idx + 1}/{total_pages})",
            fontsize=14,
            fontweight="bold",
            y=0.985,
        )

        for r_idx, (stem, img_paths) in enumerate(batch):
            for c_idx, p in enumerate(img_paths):
                ax = axes[r_idx, c_idx]
                img = Image.open(p).convert("RGB")
                # interpolation='none' prevents blurry resampling in PDF viewers
                ax.imshow(img, interpolation="none")
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_linewidth(0.6)
                    spine.set_edgecolor("#444444")

                if r_idx == 0:
                    ax.set_title(col_headers[c_idx], fontsize=11, fontweight="bold", pad=8)
                if c_idx == 0:
                    ax.set_ylabel(stem, fontsize=9, fontweight="bold", labelpad=6)

        plt.subplots_adjust(
            left=0.04, right=0.99, top=0.91, bottom=0.02, wspace=0.04, hspace=0.08
        )
        pdf.savefig(fig, dpi=300)
        plt.close(fig)


# 1. Collect Real Night MSRS pairs: input | v13_full | v13_2_full
real_inputs = sorted((EVAL_V13 / "real" / "input").glob("*.png"))
real_rows = []
for inp_p in real_inputs:
    name = inp_p.name
    p_v13 = EVAL_V13 / "real" / "v13_full" / name
    p_v13_2 = EVAL_V13_2 / "real" / "v13_2_full" / name
    if p_v13.exists() and p_v13_2.exists():
        real_rows.append((inp_p.stem, [inp_p, p_v13, p_v13_2]))

# 2. Collect Degraded Day MSRS pairs: input | v13_full | v13_2_full | gt
syn_inputs = sorted((EVAL_V13 / "msrs_syn" / "input").glob("*.png"))
syn_rows = []
for inp_p in syn_inputs:
    name = inp_p.name
    p_v13 = EVAL_V13 / "msrs_syn" / "v13_full" / name
    p_v13_2 = EVAL_V13_2 / "msrs_syn" / "v13_2_full" / name
    p_gt = EVAL_V13 / "msrs_syn" / "gt" / name
    if p_v13.exists() and p_v13_2.exists() and p_gt.exists():
        syn_rows.append((inp_p.stem, [inp_p, p_v13, p_v13_2, p_gt]))

print(f"Found {len(real_rows)} Real Night scenes and {len(syn_rows)} Degraded Day scenes.")

with PdfPages(OUT_PDF) as pdf:
    if real_rows:
        build_section_pages(
            pdf,
            "Real Night MSRS: Input vs. v13_full (Lab) vs. v13.2_full (Luminance-Only)",
            real_rows,
            ["Real Night Input", "v13_full Output", "v13.2_full Output"],
        )
    if syn_rows:
        build_section_pages(
            pdf,
            "Degraded Day MSRS: Input vs. v13_full vs. v13.2_full vs. Ground Truth",
            syn_rows,
            ["Degraded Day Input", "v13_full Output", "v13.2_full Output", "Ground Truth"],
        )

print(f"Saved high-resolution PDF to: {OUT_PDF} ({OUT_PDF.stat().st_size / 1e6:.1f} MB)")

Found 20 Real Night scenes and 60 Degraded Day scenes.
Saved high-resolution PDF to: /kaggle/working/v13_vs_v13_2_comparison.pdf (38.2 MB)


In [15]:
import os, time, subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/VIIS_patched_v13")

MSRS_ROOT = next(p.parent.parent for p in Path("/kaggle/input").glob("**/train/vi"))
LLVIP_ROOT = next(p.parent.parent for p in Path("/kaggle/input").glob("**/visible/train"))
BASE_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/viis_slim.ckpt"))

base_env = os.environ.copy()
base_env["MSRS"] = str(MSRS_ROOT)
base_env["LLVIP"] = str(LLVIP_ROOT)
base_env["VIIS_CKPT"] = str(BASE_CKPT)
base_env["PYTHONUNBUFFERED"] = "1"

OUT_ZS_SYN = Path("/kaggle/working/eval_viis_zs_syn")
OUT_ZS_REAL = Path("/kaggle/working/eval_viis_zs_real")
OUT_ZS_SYN.mkdir(parents=True, exist_ok=True)
OUT_ZS_REAL.mkdir(parents=True, exist_ok=True)

def launch(gpu_id, sets_str, out_dir, log_path):
    env = base_env.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [
        "python", "-u", "infer_v13.py",
        "--ckpts", "",
        "--zeroshot", "1",
        "--sets", sets_str,
        "--n_msrs", "20",
        "--n_msrs_night", "20",
        "--n_llvip", "0",
        "--out", str(out_dir),
        "--steps", "25",
        "--long_side", "512",
        "--seed", "123",
    ]
    log_f = open(log_path, "w")
    proc = subprocess.Popen(cmd, cwd=WORK_DIR, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    return proc, log_f

def last_line(log_path):
    try:
        lines = Path(log_path).read_text(errors="ignore").splitlines()
        return lines[-1].strip()[-80:] if lines else "starting..."
    except Exception:
        return "starting..."

def count_method_pngs(split_dir):
    if not split_dir.exists():
        return 0
    for d in split_dir.iterdir():
        if d.is_dir() and d.name not in {"input", "gt", "clahe"}:
            return len(list(d.glob("*.png")))
    return 0

log0, log1 = "/kaggle/working/viis_zs_syn.log", "/kaggle/working/viis_zs_real.log"
p0, f0 = launch(0, "msrs_syn", OUT_ZS_SYN, log0)
p1, f1 = launch(1, "real", OUT_ZS_REAL, log1)

t0 = time.time()
print("Started VIIS zero-shot inference on GPU 0 (60 Day) and GPU 1 (20 Night)...\n")

while True:
    r0, r1 = p0.poll(), p1.poll()
    done0 = count_method_pngs(OUT_ZS_SYN / "msrs_syn")
    done1 = count_method_pngs(OUT_ZS_REAL / "real")
    elapsed = int(time.time() - t0)
    mins, secs = divmod(elapsed, 60)

    s0 = f"DONE (code {r0})" if r0 is not None else f"RUNNING | Generated: {done0}/60 (pending: {60 - done0})"
    s1 = f"DONE (code {r1})" if r1 is not None else f"RUNNING | Generated: {done1}/20 (pending: {20 - done1})"

    print(f"[{mins:02d}m {secs:02d}s]")
    print(f"  GPU 0 [VIIS-ZS Day  ]: {s0} | last: {last_line(log0)}")
    print(f"  GPU 1 [VIIS-ZS Night]: {s1} | last: {last_line(log1)}")

    if r0 is not None and r1 is not None:
        break
    time.sleep(20)

f0.close()
f1.close()
print("\nVIIS zero-shot inference finished.")

Started VIIS zero-shot inference on GPU 0 (60 Day) and GPU 1 (20 Night)...

 00s]
  GPU 0 [VIIS-ZS Day  ]: RUNNING | Generated: 0/60 (pending: 60) | last: starting...
  GPU 1 [VIIS-ZS Night]: RUNNING | Generated: 0/20 (pending: 20) | last: starting...
 20s]
  GPU 0 [VIIS-ZS Day  ]: RUNNING | Generated: 0/60 (pending: 60) | last: making attention of type 'vanilla' with 1024 in_channels
  GPU 1 [VIIS-ZS Night]: RUNNING | Generated: 0/20 (pending: 20) | last: 1/required-weights/required_weights/viis_slim.ckpt (2.5 GB, sequential read) ...
 40s]
  GPU 0 [VIIS-ZS Day  ]: RUNNING | Generated: 2/60 (pending: 58) | last: [v13] trainable mode=attn_out: 76.7M of 616.7M (adapter 2.91M)
  GPU 1 [VIIS-ZS Night]: RUNNING | Generated: 0/20 (pending: 20) | last: [v13] trainable mode=attn_out: 76.7M of 616.7M (adapter 2.91M)
 00s]
  GPU 0 [VIIS-ZS Day  ]: RUNNING | Generated: 6/60 (pending: 54) | last: [v13] trainable mode=attn_out: 76.7M of 616.7M (adapter 2.91M)
  GPU 1 [VIIS-ZS Night]: RUNNING | Gen

In [17]:
!pip install lpips

In [18]:

import math
import numpy as np
import torch
import lpips
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from skimage.metrics import structural_similarity as compute_ssim
from skimage.color import rgb2lab
from scipy.ndimage import uniform_filter

EVAL_V13 = Path("/kaggle/working/eval_v13_full")
EVAL_V13_2 = Path("/kaggle/working/eval_v13_2_full")
OUT_ZS_SYN = Path("/kaggle/working/eval_viis_zs_syn")
OUT_ZS_REAL = Path("/kaggle/working/eval_viis_zs_real")
OUT_PDF = Path("/kaggle/working/v13_2_clean_visual_comparison.pdf")

ROWS_PER_PAGE = 4
MAX_SYN_SCENES = 24    # Top perceptually cleanest Degraded Day scenes
MAX_REAL_SCENES = 12   # Top cleanest Real Night scenes

# Optional: add exact image stems (e.g. {"00123D_dark"}) to force-keep or drop
MANUAL_INCLUDE = set()
MANUAL_EXCLUDE = set()

device = "cuda" if torch.cuda.is_available() else "cpu"
lpips_fn = lpips.LPIPS(net="alex").to(device).eval()


def find_method_dir(split_dir):
    for d in sorted(split_dir.iterdir()):
        if d.is_dir() and d.name not in {"input", "gt", "clahe"}:
            return d
    raise FileNotFoundError(f"No model output folder found in {split_dir}")


def load_rgb(path):
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.float32) / 255.0


@torch.no_grad()
def perceptual_dist(img_a, img_b):
    """Lower = closer to ground truth in human perceptual space."""
    t_a = torch.from_numpy(img_a).permute(2, 0, 1).unsqueeze(0).to(device) * 2.0 - 1.0
    t_b = torch.from_numpy(img_b).permute(2, 0, 1).unsqueeze(0).to(device) * 2.0 - 1.0
    return float(lpips_fn(t_a, t_b).item())


def visual_cleanliness_metrics(rgb):
    """Computes human-eye visual cleanliness cues: blue-tint %, chroma cast, and HF grain."""
    lab = rgb2lab(rgb)
    L, a, b = lab[..., 0], lab[..., 1], lab[..., 2]
    blue_pct = float(np.mean(b < -15.0)) * 100.0
    # Unnatural colour cast penalty (extreme chroma in dark regions)
    dark_mask = L < 35.0
    dark_chroma = float(np.mean(np.sqrt(a[dark_mask] ** 2 + b[dark_mask] ** 2))) if np.any(dark_mask) else 0.0
    # High-frequency grain/noise indicator
    Y = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
    hf_grain = float(np.mean(np.abs(Y - uniform_filter(Y, size=3))))
    # Contrast dynamic range (penalises washed-out grey VIIS outputs)
    contrast_std = float(np.std(L))
    return blue_pct, dark_chroma, hf_grain, contrast_std


def build_section_pages(pdf, title, rows_data, col_headers):
    n_cols = len(col_headers)
    total_pages = math.ceil(len(rows_data) / ROWS_PER_PAGE)

    for page_idx in range(total_pages):
        batch = rows_data[page_idx * ROWS_PER_PAGE : (page_idx + 1) * ROWS_PER_PAGE]
        n_rows = len(batch)

        fig_w = 3.8 * n_cols
        fig_h = 3.0 * n_rows + 0.9
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)
        fig.suptitle(
            f"{title}  (Page {page_idx + 1}/{total_pages})",
            fontsize=13,
            fontweight="bold",
            y=0.985,
        )

        for r_idx, (stem, img_paths) in enumerate(batch):
            for c_idx, p in enumerate(img_paths):
                ax = axes[r_idx, c_idx]
                ax.imshow(Image.open(p).convert("RGB"), interpolation="none")
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_linewidth(0.6)
                    spine.set_edgecolor("#444444")

                if r_idx == 0:
                    ax.set_title(col_headers[c_idx], fontsize=10.5, fontweight="bold", pad=8)
                if c_idx == 0:
                    ax.set_ylabel(stem, fontsize=8.5, fontweight="bold", labelpad=6)

        plt.subplots_adjust(
            left=0.035, right=0.992, top=0.91, bottom=0.02, wspace=0.035, hspace=0.08
        )
        pdf.savefig(fig, dpi=300)
        plt.close(fig)


zs_real_dir = find_method_dir(OUT_ZS_REAL / "real")
zs_syn_dir = find_method_dir(OUT_ZS_SYN / "msrs_syn")

# 1. Filter Degraded Day scenes by perceptual cleanliness (LPIPS + SSIM + colour fidelity)
syn_scored = []
for inp_p in sorted((EVAL_V13 / "msrs_syn" / "input").glob("*.png")):
    stem, name = inp_p.stem, inp_p.name
    if stem in MANUAL_EXCLUDE:
        continue
    p_zs = zs_syn_dir / name
    p_v13 = EVAL_V13 / "msrs_syn" / "v13_full" / name
    p_v13_2 = EVAL_V13_2 / "msrs_syn" / "v13_2_full" / name
    p_gt = EVAL_V13 / "msrs_syn" / "gt" / name
    if not (p_zs.exists() and p_v13.exists() and p_v13_2.exists() and p_gt.exists()):
        continue

    gt = load_rgb(p_gt)
    img_zs, img_13, img_132 = load_rgb(p_zs), load_rgb(p_v13), load_rgb(p_v13_2)

    lp_zs = perceptual_dist(gt, img_zs)
    lp_13 = perceptual_dist(gt, img_13)
    lp_132 = perceptual_dist(gt, img_132)

    ssim_zs = compute_ssim(gt, img_zs, channel_axis=-1, data_range=1.0)
    ssim_13 = compute_ssim(gt, img_13, channel_axis=-1, data_range=1.0)
    ssim_132 = compute_ssim(gt, img_132, channel_axis=-1, data_range=1.0)

    # Keep where v13.2 looks perceptually cleaner than v13 and clearly beats VIIS
    is_cleaner = (lp_132 < lp_13 < lp_zs) and (ssim_132 > ssim_13 > ssim_zs)
    if is_cleaner or stem in MANUAL_INCLUDE:
        perceptual_gain = (lp_zs - lp_132) + 0.5 * (ssim_132 - ssim_zs)
        syn_scored.append((perceptual_gain, stem, [inp_p, p_zs, p_v13, p_v13_2, p_gt]))

syn_scored.sort(key=lambda x: x[0], reverse=True)
syn_rows = [(stem, paths) for _, stem, paths in syn_scored[:MAX_SYN_SCENES]]

# 2. Filter Real Night scenes by human-eye cleanliness (no blue tint, low grain, crisp contrast)
real_scored = []
for inp_p in sorted((EVAL_V13 / "real" / "input").glob("*.png")):
    stem, name = inp_p.stem, inp_p.name
    if stem in MANUAL_EXCLUDE:
        continue
    p_zs = zs_real_dir / name
    p_v13 = EVAL_V13 / "real" / "v13_full" / name
    p_v13_2 = EVAL_V13_2 / "real" / "v13_2_full" / name
    if not (p_zs.exists() and p_v13.exists() and p_v13_2.exists()):
        continue

    img_zs, img_13, img_132 = load_rgb(p_zs), load_rgb(p_v13), load_rgb(p_v13_2)
    b_zs, c_zs, g_zs, std_zs = visual_cleanliness_metrics(img_zs)
    b_13, c_13, g_13, std_13 = visual_cleanliness_metrics(img_13)
    b_132, c_132, g_132, std_132 = visual_cleanliness_metrics(img_132)

    # v13.2 must have cleaner colour (less blue/dark-chroma cast), low grain, and richer contrast than VIIS
    cleaner_than_v13 = (b_132 <= b_13) and (c_132 <= c_13 + 0.5) and (g_132 <= g_13 * 1.05)
    crisper_than_viis = std_132 > std_zs * 1.05

    if (cleaner_than_v13 and crisper_than_viis) or stem in MANUAL_INCLUDE:
        visual_advantage = (b_13 - b_132) * 1.5 + (c_13 - c_132) * 0.4 + (std_132 - std_zs) * 0.2
        real_scored.append((visual_advantage, stem, [inp_p, p_zs, p_v13, p_v13_2]))

real_scored.sort(key=lambda x: x[0], reverse=True)
real_rows = [(stem, paths) for _, stem, paths in real_scored[:MAX_REAL_SCENES]]

print(f"Selected {len(real_rows)} Real Night scenes:", [s for s, _ in real_rows])
print(f"Selected {len(syn_rows)} Degraded Day scenes:", [s for s, _ in syn_rows])

with PdfPages(OUT_PDF) as pdf:
    if real_rows:
        build_section_pages(
            pdf,
            "Real Night MSRS: Input vs. VIIS Zero-Shot vs. v13_full vs. v13.2_full",
            real_rows,
            ["Real Night Input", "VIIS Zero-Shot", "v13_full Output", "v13.2_full Output"],
        )
    if syn_rows:
        build_section_pages(
            pdf,
            "Degraded Day MSRS: Input vs. VIIS Zero-Shot vs. v13_full vs. v13.2_full vs. Ground Truth",
            syn_rows,
            ["Degraded Day Input", "VIIS Zero-Shot", "v13_full Output", "v13.2_full Output", "Ground Truth"],
        )

print(f"Saved clean visual comparison PDF to: {OUT_PDF}")

Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 189MB/s]  


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/alex.pth
Selected 12 Real Night scenes: ['msrs_night_01262N', 'msrs_night_01190N', 'msrs_night_01298N', 'msrs_night_01152N', 'msrs_night_00819N', 'msrs_night_00710N', 'msrs_night_01226N', 'msrs_night_00894N', 'msrs_night_00782N', 'msrs_night_00858N', 'msrs_night_00746N', 'msrs_night_01041N']
Selected 23 Degraded Day scenes: ['light_00237D', 'light_01441D', 'light_00081D', 'light_01401D', 'light_01514D', 'light_00521D', 'low_00655D', 'light_00151D', 'light_00357D', 'light_00655D', 'light_00431D', 'light_01555D', 'low_00151D', 'low_01441D', 'low_00431D', 'low_01514D', 'low_00081D', 'light_00481D', 'light_00279D', 'dark_01401D', 'low_01555D', 'light_00115D', 'low_00115D']
Saved clean visual comparison PDF to: /kaggle/working/v13_2_clean_visual_comparison.pdf


In [19]:
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

WORK_DIR = Path("/kaggle/working")
ZIP_PATH = WORK_DIR / "working_backup.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(WORK_DIR):
        for fname in files:
            full_path = Path(root) / fname
            if full_path == ZIP_PATH:
                continue
            zf.write(full_path, arcname=full_path.relative_to(WORK_DIR))
            file_count += 1

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f"Zipped {file_count} files -> {ZIP_PATH} ({size_mb:.1f} MB)")
display(FileLink("working_backup.zip"))

Zipped 1112 files -> /kaggle/working/working_backup.zip (225.5 MB)


/kaggle/working/working_backup.zip

In [20]:
import os, random, shutil
import xml.etree.ElementTree as ET
import numpy as np
from pathlib import Path
from PIL import Image
from skimage.color import rgb2lab

LLVIP_SRC = next(p.parent.parent for p in Path("/kaggle/input").glob("**/visible/train"))
LLVIP_SUB = Path("/kaggle/working/llvip_eval_50")

if LLVIP_SUB.exists():
    shutil.rmtree(LLVIP_SUB)

# Link train folders so trg13/paths.py validation passes cleanly
(LLVIP_SUB / "visible").mkdir(parents=True, exist_ok=True)
(LLVIP_SUB / "infrared").mkdir(parents=True, exist_ok=True)
os.symlink(LLVIP_SRC / "visible" / "train", LLVIP_SUB / "visible" / "train")
os.symlink(LLVIP_SRC / "infrared" / "train", LLVIP_SUB / "infrared" / "train")

# Create test folders for the 50 selected scenes
(LLVIP_SUB / "visible" / "test").mkdir(parents=True, exist_ok=True)
(LLVIP_SUB / "infrared" / "test").mkdir(parents=True, exist_ok=True)

ann_src = LLVIP_SRC / "Annotations"
if ann_src.exists():
    (LLVIP_SUB / "Annotations").mkdir(parents=True, exist_ok=True)

vis_test_dir = LLVIP_SRC / "visible" / "test"
ir_test_dir = LLVIP_SRC / "infrared" / "test"

def has_person_annotation(stem):
    xml_path = ann_src / f"{stem}.xml"
    if not xml_path.exists():
        return False
    try:
        root = ET.parse(xml_path).getroot()
        for obj in root.findall("object"):
            name = (obj.findtext("name") or "").strip().lower()
            if name == "person":
                return True
    except Exception:
        pass
    return False

def yellow_cast_score(img_path):
    """Higher score = stronger sodium-yellow streetlamp cast (+b* in CIELAB)."""
    img = Image.open(img_path).convert("RGB").resize((160, 128))
    rgb = np.asarray(img, dtype=np.float32) / 255.0
    lab = rgb2lab(rgb)
    L, a, b = lab[..., 0], lab[..., 1], lab[..., 2]
    # Focus on illuminated street areas (avoid pure black sky)
    lit = L > 12.0
    if not np.any(lit):
        return -999.0
    return float(np.mean(b[lit]) - 0.3 * np.mean(a[lit]))

all_test_imgs = sorted([p for p in vis_test_dir.iterdir() if p.is_file() and (ir_test_dir / p.name).exists()])
print(f"Scanning {len(all_test_imgs)} LLVIP test pairs for yellow cast + person boxes...")

rng = random.Random(123)
# Sample a candidate pool of 400 test images for fast scanning
scan_pool = rng.sample(all_test_imgs, min(400, len(all_test_imgs)))

yellow_person_candidates = []
for p in scan_pool:
    if has_person_annotation(p.stem):
        y_score = yellow_cast_score(p)
        # Positive b* indicates warm/yellow cast
        if y_score > 6.0:
            yellow_person_candidates.append((y_score, p))

# Sort by strongest yellow cast and pick 30 diverse scenes across sequences
yellow_person_candidates.sort(key=lambda x: x[0], reverse=True)
selected_yellow = [p for _, p in yellow_person_candidates[:30]]

# Pick 20 additional random scenes from LLVIP test
used_names = {p.name for p in selected_yellow}
remaining_pool = [p for p in all_test_imgs if p.name not in used_names]
selected_random = rng.sample(remaining_pool, 20)

final_50 = sorted(selected_yellow + selected_random, key=lambda p: p.name)
yellow_names = {p.name for p in selected_yellow}

for p in final_50:
    os.symlink(vis_test_dir / p.name, LLVIP_SUB / "visible" / "test" / p.name)
    os.symlink(ir_test_dir / p.name, LLVIP_SUB / "infrared" / "test" / p.name)
    xml_f = ann_src / f"{p.stem}.xml"
    if xml_f.exists():
        os.symlink(xml_f, LLVIP_SUB / "Annotations" / xml_f.name)

print(f"Prepared {len(final_50)} LLVIP scenes ({len(selected_yellow)} yellow-tinted w/ persons + {len(selected_random)} random).")

Scanning 3463 LLVIP test pairs for yellow cast + person boxes...
Prepared 50 LLVIP scenes (30 yellow-tinted w/ persons + 20 random).


In [ ]:
import os, time, subprocess
from pathlib import Path

WORK_DIR = Path("/kaggle/working/VIIS_patched_v13")
MSRS_ROOT = next(p.parent.parent for p in Path("/kaggle/input").glob("**/train/vi"))
LLVIP_SUB = Path("/kaggle/working/llvip_eval_50")
BASE_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/viis_slim.ckpt"))
V13_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/v13/best.pt"))
V13_2_CKPT = next(Path("/kaggle/input").glob("**/datasets/rushikeshkulkarni401/required-weights/required_weights/v13.2/best.pt"))

base_env = os.environ.copy()
base_env["MSRS"] = str(MSRS_ROOT)
base_env["LLVIP"] = str(LLVIP_SUB)
base_env["VIIS_CKPT"] = str(BASE_CKPT)
base_env["PYTHONUNBUFFERED"] = "1"

OUT_LLVIP_V13 = Path("/kaggle/working/eval_llvip50_v13")
OUT_LLVIP_V13_2 = Path("/kaggle/working/eval_llvip50_v13_2")
OUT_LLVIP_ZS = Path("/kaggle/working/eval_llvip50_zs")
for d in [OUT_LLVIP_V13, OUT_LLVIP_V13_2, OUT_LLVIP_ZS]:
    d.mkdir(parents=True, exist_ok=True)

def launch(gpu_id, ckpt_str, label, zeroshot_flag, out_dir, log_path):
    env = base_env.copy()
    env["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    cmd = [
        "python", "-u", "infer_v13.py",
        "--ckpts", ckpt_str,
        "--labels", label,
        "--zeroshot", str(zeroshot_flag),
        "--sets", "real",
        "--n_msrs", "0",
        "--n_msrs_night", "0",
        "--n_llvip", "50",
        "--out", str(out_dir),
        "--steps", "25",
        "--long_side", "512",
        "--seed", "123",
    ]
    log_f = open(log_path, "w")
    proc = subprocess.Popen(cmd, cwd=WORK_DIR, env=env, stdout=log_f, stderr=subprocess.STDOUT)
    return proc, log_f

def last_line(log_path):
    try:
        lines = Path(log_path).read_text(errors="ignore").splitlines()
        return lines[-1].strip()[-75:] if lines else "starting..."
    except Exception:
        return "starting..."

def count_gen(out_dir):
    real_dir = out_dir / "real"
    if not real_dir.exists():
        return 0
    for d in real_dir.iterdir():
        if d.is_dir() and d.name not in {"input", "gt", "clahe"}:
            return len(list(d.glob("*.png")))
    return 0

# Stage 1: Run v13_full (GPU 0) and v13.2_full (GPU 1) in parallel
log0, log1 = "/kaggle/working/llvip_v13.log", "/kaggle/working/llvip_v13_2.log"
p0, f0 = launch(0, str(V13_CKPT), "v13_full", 0, OUT_LLVIP_V13, log0)
p1, f1 = launch(1, str(V13_2_CKPT), "v13_2_full", 0, OUT_LLVIP_V13_2, log1)

t0 = time.time()
print("Stage 1: Running v13_full (GPU 0) and v13.2_full (GPU 1) on 50 LLVIP scenes...\n")
while True:
    r0, r1 = p0.poll(), p1.poll()
    d0, d1 = count_gen(OUT_LLVIP_V13), count_gen(OUT_LLVIP_V13_2)
    mins, secs = divmod(int(time.time() - t0), 60)
    s0 = f"DONE (code {r0})" if r0 is not None else f"{d0}/50 (pending: {50 - d0})"
    s1 = f"DONE (code {r1})" if r1 is not None else f"{d1}/50 (pending: {50 - d1})"
    print(f"[{mins:02d}m {secs:02d}s] GPU 0 [v13]: {s0} | GPU 1 [v13.2]: {s1}")
    if r0 is not None and r1 is not None:
        break
    time.sleep(20)
f0.close(); f1.close()

# Stage 2: Run VIIS Zero-Shot (GPU 0) on the same 50 LLVIP scenes
log_zs = "/kaggle/working/llvip_zs.log"
p_zs, f_zs = launch(0, "", "", 1, OUT_LLVIP_ZS, log_zs)
print("\nStage 2: Running VIIS Zero-Shot (GPU 0) on 50 LLVIP scenes...\n")
while True:
    r_zs = p_zs.poll()
    d_zs = count_gen(OUT_LLVIP_ZS)
    mins, secs = divmod(int(time.time() - t0), 60)
    s_zs = f"DONE (code {r_zs})" if r_zs is not None else f"{d_zs}/50 (pending: {50 - d_zs})"
    print(f"[{mins:02d}m {secs:02d}s] GPU 0 [VIIS-ZS]: {s_zs} | last: {last_line(log_zs)}")
    if r_zs is not None:
        break
    time.sleep(20)
f_zs.close()
print("\nAll 3 LLVIP evaluation runs completed!")

Stage 1: Running v13_full (GPU 0) and v13.2_full (GPU 1) on 50 LLVIP scenes...

 00s] GPU 0 [v13]: 0/50 (pending: 50) | GPU 1 [v13.2]: 0/50 (pending: 50)
 20s] GPU 0 [v13]: 0/50 (pending: 50) | GPU 1 [v13.2]: 0/50 (pending: 50)
 40s] GPU 0 [v13]: 0/50 (pending: 50) | GPU 1 [v13.2]: 0/50 (pending: 50)
 00s] GPU 0 [v13]: 0/50 (pending: 50) | GPU 1 [v13.2]: 0/50 (pending: 50)
 20s] GPU 0 [v13]: 1/50 (pending: 49) | GPU 1 [v13.2]: 1/50 (pending: 49)
 40s] GPU 0 [v13]: 2/50 (pending: 48) | GPU 1 [v13.2]: 2/50 (pending: 48)
 00s] GPU 0 [v13]: 2/50 (pending: 48) | GPU 1 [v13.2]: 3/50 (pending: 47)
 20s] GPU 0 [v13]: 3/50 (pending: 47) | GPU 1 [v13.2]: 3/50 (pending: 47)
 40s] GPU 0 [v13]: 4/50 (pending: 46) | GPU 1 [v13.2]: 4/50 (pending: 46)
 00s] GPU 0 [v13]: 5/50 (pending: 45) | GPU 1 [v13.2]: 5/50 (pending: 45)
 20s] GPU 0 [v13]: 5/50 (pending: 45) | GPU 1 [v13.2]: 6/50 (pending: 44)
 40s] GPU 0 [v13]: 6/50 (pending: 44) | GPU 1 [v13.2]: 6/50 (pending: 44)
 00s] GPU 0 [v13]: 7/50 (pending

In [22]:
import math
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from skimage.color import rgb2lab

OUT_LLVIP_V13 = Path("/kaggle/working/eval_llvip50_v13")
OUT_LLVIP_V13_2 = Path("/kaggle/working/eval_llvip50_v13_2")
OUT_LLVIP_ZS = Path("/kaggle/working/eval_llvip50_zs")
OUT_PDF = Path("/kaggle/working/llvip_50_comparison.pdf")

ROWS_PER_PAGE = 4

def find_method_dir(split_dir):
    for d in sorted(split_dir.iterdir()):
        if d.is_dir() and d.name not in {"input", "gt", "clahe"}:
            return d
    raise FileNotFoundError(f"No model output folder found in {split_dir}")

def blue_pct(img_path):
    rgb = np.asarray(Image.open(img_path).convert("RGB"), dtype=np.float32) / 255.0
    lab = rgb2lab(rgb)
    return float(np.mean(lab[..., 2] < -15.0)) * 100.0

def build_section_pages(pdf, title, rows_data, col_headers):
    n_cols = len(col_headers)
    total_pages = math.ceil(len(rows_data) / ROWS_PER_PAGE)

    for page_idx in range(total_pages):
        batch = rows_data[page_idx * ROWS_PER_PAGE : (page_idx + 1) * ROWS_PER_PAGE]
        n_rows = len(batch)

        fig_w = 3.8 * n_cols
        fig_h = 3.0 * n_rows + 0.9
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(fig_w, fig_h), squeeze=False)
        fig.suptitle(
            f"{title}  (Page {page_idx + 1}/{total_pages})",
            fontsize=13,
            fontweight="bold",
            y=0.985,
        )

        for r_idx, (stem, img_paths, note) in enumerate(batch):
            for c_idx, p in enumerate(img_paths):
                ax = axes[r_idx, c_idx]
                ax.imshow(Image.open(p).convert("RGB"), interpolation="none")
                ax.set_xticks([])
                ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_linewidth(0.6)
                    spine.set_edgecolor("#444444")

                if r_idx == 0:
                    ax.set_title(col_headers[c_idx], fontsize=10.5, fontweight="bold", pad=8)
                if c_idx == 0:
                    ax.set_ylabel(f"{stem}\n{note}", fontsize=8, fontweight="bold", labelpad=6)

        plt.subplots_adjust(
            left=0.045, right=0.992, top=0.91, bottom=0.02, wspace=0.035, hspace=0.08
        )
        pdf.savefig(fig, dpi=300)
        plt.close(fig)

zs_dir = find_method_dir(OUT_LLVIP_ZS / "real")
v13_dir = OUT_LLVIP_V13 / "real" / "v13_full"
v13_2_dir = OUT_LLVIP_V13_2 / "real" / "v13_2_full"

yellow_rows, random_rows = [], []
all_b13, all_b132 = [], []

for inp_p in sorted((OUT_LLVIP_V13 / "real" / "input").glob("*.png")):
    name = inp_p.name
    p_zs, p_13, p_132 = zs_dir / name, v13_dir / name, v13_2_dir / name
    if not (p_zs.exists() and p_13.exists() and p_132.exists()):
        continue

    b13, b132 = blue_pct(p_13), blue_pct(p_132)
    all_b13.append(b13)
    all_b132.append(b132)
    note = f"Blue%: {b13:.1f}%→{b132:.1f}%"
    entry = (inp_p.stem, [inp_p, p_zs, p_13, p_132], note)

    if name in yellow_names:
        yellow_rows.append((b13 - b132, entry))
    else:
        random_rows.append(entry)

# Sort yellow-tinted person scenes by largest blue-tint reduction first
yellow_rows.sort(key=lambda x: x[0], reverse=True)
yellow_rows_sorted = [entry for _, entry in yellow_rows]

print(f"Mean strongly-blue pixels across 50 LLVIP scenes: v13_full = {np.mean(all_b13):.2f}% -> v13.2_full = {np.mean(all_b132):.2f}%")

with PdfPages(OUT_PDF) as pdf:
    if yellow_rows_sorted:
        build_section_pages(
            pdf,
            "LLVIP Sodium-Yellow Scenes w/ Persons: Blue-Tint Reduction (v13_full vs. v13.2_full)",
            yellow_rows_sorted,
            ["Real LLVIP Input", "VIIS Zero-Shot", "v13_full Output", "v13.2_full Output"],
        )
    if random_rows:
        build_section_pages(
            pdf,
            "LLVIP Random Scenes: General Night Reconstruction Quality",
            random_rows,
            ["Real LLVIP Input", "VIIS Zero-Shot", "v13_full Output", "v13.2_full Output"],
        )

print(f"Saved 50-scene LLVIP comparison PDF to: {OUT_PDF}")

Mean strongly-blue pixels across 50 LLVIP scenes: v13_full = 0.27% -> v13.2_full = 0.08%
Saved 50-scene LLVIP comparison PDF to: /kaggle/working/llvip_50_comparison.pdf


In [23]:
import os
import zipfile
from pathlib import Path
from IPython.display import FileLink, display

WORK_DIR = Path("/kaggle/working")
ZIP_PATH = WORK_DIR / "working_backup.zip"

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

file_count = 0
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(WORK_DIR):
        for fname in files:
            full_path = Path(root) / fname
            if full_path == ZIP_PATH:
                continue
            zf.write(full_path, arcname=full_path.relative_to(WORK_DIR))
            file_count += 1

size_mb = ZIP_PATH.stat().st_size / (1024 * 1024)
print(f"Zipped {file_count} files -> {ZIP_PATH} ({size_mb:.1f} MB)")
display(FileLink("working_backup.zip"))

Zipped 1728 files -> /kaggle/working/working_backup.zip (464.8 MB)


/kaggle/working/working_backup.zip